# Reconstruction Visualization

Hamburg 1024×1024 tile visual comparison across architectures (ResSHyp / FP) and backends
(GPU FP32 / FPGA INT8) for selected λ values.

Data sources:
- **GPU** — FP32 reconstructions from W&B run directories (`run_dir`).
- **FPGA** — INT8 reconstructions from `compiled_models/<name>/results/<tile>_recon_linA.npy`.

All images shown as log-intensity, clipped per-image to mean ± 3σ.

In [ ]:
import json
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import torch

ROOT_DIR = Path("..").resolve()
sys.path.insert(0, str(ROOT_DIR))

from src.utils.metrics import psnr as _src_psnr

# ── Paths ─────────────────────────────────────────────────────────────────────
WANDB_CSV = ROOT_DIR / "notebooks" / "SAR_DDC_FPGA_all_runs_WandB.csv"
COMPILED_MODELS_DIR = ROOT_DIR / "results" / "fpga" / "compiled_models"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "reconstruction"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
SAVE_FIGURES = True

# ── Shared conventions, palette, rendering ────────────────────────────────────
sys.path.insert(0, str(ROOT_DIR / "notebooks"))
from _plotkit import (
    PALETTE,
    ARCH_LABEL,
    BACKEND_COLORS,
    PROD_LR,
    linA_to_logI,
    show_logI,
    export_manuscript,
)

ARCH_LBL = ARCH_LABEL  # alias used throughout this notebook
ACCEPTED_SEEDS = [0, 1, 2, 3, 4, 5]

r, g, b, y, e = "\033[31m", "\033[32m", "\033[34m", "\033[33m", "\033[0m"

In [ ]:
# ── Load GPU runs (FP32) ──────────────────────────────────────────────────────
# Uses _plotkit.load_quality_runs; result includes run_dir for GPU recon .npy paths.
sys.path.insert(0, str(ROOT_DIR / "notebooks"))
from _plotkit import load_quality_runs, load_fpga_quality

gpu_df = load_quality_runs(WANDB_CSV, lr="prod", seeds=ACCEPTED_SEEDS, verbose=True)
fpga_df = load_fpga_quality(
    gpu_df=gpu_df, compiled_dir=COMPILED_MODELS_DIR, seeds=ACCEPTED_SEEDS, verbose=True
)

# ── Merge into one tidy df ────────────────────────────────────────────────────
df = pd.concat([gpu_df, fpga_df], ignore_index=True)
df["lambda"] = df["lambda"].astype(float)
df["seed"] = df["seed"].astype(int)
df = df.sort_values(["lambda", "seed", "backend"]).reset_index(drop=True)
print(f"\nCombined: {len(df)} rows ({df['backend'].value_counts().to_dict()})")

## Hamburg Tile Visualization

Visual comparison on the **1024 × 1024 Hamburg tile** across selected λ values.

**Layout** — one reference row + one FPGA/GPU row-pair per architecture in `ARCHS_FOR_VIS`:
- 1 arch → 3 rows (refs / FPGA / GPU)
- 2 archs → 5 rows (refs / FPGA arch₁ / GPU arch₁ / FPGA arch₂ / GPU arch₂)

All images are log-I, clipped per-image to mean ± 3σ.
PSNR is recomputed on-the-fly from the `.npy` arrays vs the MERLIN reference.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
TILE_NAME = "Hamburg_[11000:12024-8500:9524]"
ARCHS_FOR_VIS = ["ResSHyp", "FP"]  # ["ResSHyp", "SHyp"] → 5 rows; ["ResSHyp"] → 3 rows
SHOW_LAMBDAS = [1, 20, 1000]  # None → all 10 lambdas  (paper: 1/20/1000)
SEED_FOR_VIS = 0
SHOW_FRAMES = False  # backend-colored cell borders (paper: off)
SHOW_ADAM_NOC = False
SHOW_MERLIN_DDS = False
SHOW_CLOSEUP_ROI = True
CROP_R0, CROP_R1 = 0, 200
CROP_C0, CROP_C1 = 100, 300
# ─────────────────────────────────────────────────────────────────────────────

REFS_DIR = ROOT_DIR / "data" / "visualization" / TILE_NAME
SUBPLOT_IN = 2.6  # inches per subplot

# ── Helpers ───────────────────────────────────────────────────────────────────
# Use _plotkit implementations; alias local names for backward compat in rc-05/rc-07.
_linA_to_logI = linA_to_logI
_show = show_logI


def _compute_psnr_tile(recon: np.ndarray, ref: np.ndarray) -> float:
    r_ = torch.from_numpy(recon.astype(np.float32))
    t_ = torch.from_numpy(ref.astype(np.float32))
    return _src_psnr(r_, t_)


def _crop(arr: np.ndarray) -> np.ndarray:
    return arr[CROP_R0:CROP_R1, CROP_C0:CROP_C1]


# ── Load reference images ─────────────────────────────────────────────────────
noisy_linA = np.load(REFS_DIR / "linA_Noisy.npy")
merlin_linA = np.load(REFS_DIR / "linA_MERLIN.npy")
adam_noc_linA = np.load(REFS_DIR / "linA_ADAM_NOC.npy") if SHOW_ADAM_NOC else None
merlin_dds_linA = np.load(REFS_DIR / "linA_MERLIN_DDS.npy") if SHOW_MERLIN_DDS else None

ref_panels: List[Tuple[str, np.ndarray]] = [("Noisy", noisy_linA), ("MERLIN", merlin_linA)]
if adam_noc_linA is not None:
    ref_panels.append(("ADAM-NOC", adam_noc_linA))
if merlin_dds_linA is not None:
    ref_panels.append(("MERLIN-DDS", merlin_dds_linA))


# ── Load FPGA & GPU reconstructions ──────────────────────────────────────────
sel_lambdas: List[float] = sorted(SHOW_LAMBDAS) if SHOW_LAMBDAS else sorted(df["lambda"].unique())

tile_data: Dict[str, Dict[float, Dict[str, Optional[Dict]]]] = {}

for _arch in ARCHS_FOR_VIS:
    tile_data[_arch] = {}
    for lmbda in sel_lambdas:
        tile_data[_arch][lmbda] = {}
        for backend in ["fpga", "gpu"]:
            rows = df[
                (df["lambda"] == lmbda)
                & (df["seed"] == SEED_FOR_VIS)
                & (df["backend"] == backend)
                & (df["arch"] == _arch)
            ]
            if len(rows) == 0:
                tile_data[_arch][lmbda][backend] = None
                continue
            row = rows.iloc[0]

            if backend == "gpu":
                if not row["run_dir"]:
                    raise ValueError(
                        f"GPU run for λ={int(lmbda)} seed={SEED_FOR_VIS} arch={_arch} has no "
                        "run_dir. Re-evaluate the run to generate Hamburg tile reconstructions."
                    )
                npy_path = Path(row["run_dir"]) / f"recon_{TILE_NAME}_linA.npy"
                metrics_path = Path(row["run_dir"]) / f"recon_{TILE_NAME}_metrics.json"
            else:
                npy_path = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_recon_linA.npy"
                metrics_path = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_metrics.json"

            if not npy_path.exists():
                print(
                    f"  {y}MISSING{e}: λ={int(lmbda)} seed={SEED_FOR_VIS} {backend} {_arch} — {npy_path.name}"
                )
                tile_data[_arch][lmbda][backend] = None
                continue

            linA = np.load(npy_path)
            psnr = _compute_psnr_tile(linA, merlin_linA)
            bpp = float("nan")
            if metrics_path.exists():
                m = json.loads(metrics_path.read_text())
                bpp = float(m.get("bpp" if backend == "fpga" else "bpp_bitstream", float("nan")))

            tile_data[_arch][lmbda][backend] = {
                "linA": linA,
                "logI": _linA_to_logI(linA),
                "psnr_merlin": psnr,
                "bpp": bpp,
            }

n_present = sum(
    1
    for arch_d in tile_data.values()
    for lam_d in arch_d.values()
    for v in lam_d.values()
    if v is not None
)
print(
    f"Loaded {n_present} / {2 * len(ARCHS_FOR_VIS) * len(sel_lambdas)} tiles  "
    f"(seed={SEED_FOR_VIS}, {len(ARCHS_FOR_VIS)} arch × {len(sel_lambdas)} λ × 2 backends)"
)

In [ ]:
n_refs = len(ref_panels)
n_lam = len(sel_lambdas)
n_cols = max(n_refs, n_lam)
n_rows = 1 + 2 * len(ARCHS_FOR_VIS)

_ROW_HEADERS = [(0, "References", "black")]
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    _suffix = f"\n{ARCH_LBL.get(_arch, _arch)}" if len(ARCHS_FOR_VIS) > 1 else ""
    _ROW_HEADERS += [
        (1 + 2 * _i, f"FPGA{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["fpga"]),
        (2 + 2 * _i, f"GPU{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["gpu"]),
    ]

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(SUBPLOT_IN * n_cols + 0.5, SUBPLOT_IN * n_rows + 0.7),
    squeeze=False,
)

for _ri, _label, _color in _ROW_HEADERS:
    axes[_ri, 0].text(
        -0.12,
        0.5,
        _label,
        transform=axes[_ri, 0].transAxes,
        rotation=90,
        va="center",
        ha="right",
        fontsize=9,
        fontweight="bold",
        color=_color,
    )

# ── Row 0: References ─────────────────────────────────────────────────────────
for ci, (ref_name, ref_linA) in enumerate(ref_panels):
    _show(axes[0, ci], _linA_to_logI(ref_linA), ref_name)
    if SHOW_CLOSEUP_ROI and ref_name == "Noisy":
        axes[0, ci].add_patch(
            Rectangle(
                (CROP_C0, CROP_R0),
                CROP_C1 - CROP_C0,
                CROP_R1 - CROP_R0,
                linewidth=1.5,
                edgecolor="red",
                facecolor="none",
                linestyle="--",
            )
        )
for ci in range(n_refs, n_cols):
    axes[0, ci].axis("off")

# ── FPGA + GPU rows ────────────────────────────────────────────────────────────
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    for _j, backend in enumerate(["fpga", "gpu"]):
        row_idx = 1 + 2 * _i + _j
        color = BACKEND_COLORS[backend]
        for ci, lmbda in enumerate(sel_lambdas):
            data = tile_data[_arch].get(lmbda, {}).get(backend)
            if data is None:
                axes[row_idx, ci].axis("off")
                axes[row_idx, ci].text(
                    0.5,
                    0.5,
                    "N/A",
                    ha="center",
                    va="center",
                    transform=axes[row_idx, ci].transAxes,
                    fontsize=10,
                    color="gray",
                )
                continue
            bpp_str = f"{data['bpp']:.3f}" if not np.isnan(data["bpp"]) else "N/A"
            subtitle = f"BPP={bpp_str}  PSNR={data['psnr_merlin']:.2f}dB"
            _show(
                axes[row_idx, ci],
                data["logI"],
                f"λ={int(lmbda)}",
                subtitle,
                border_color=(color if SHOW_FRAMES else None),
            )
        for ci in range(n_lam, n_cols):
            axes[row_idx, ci].axis("off")

arch_str = " + ".join(ARCH_LBL.get(a, a) for a in ARCHS_FOR_VIS)
fig.suptitle(
    f"Hamburg Tile  —  {TILE_NAME}\n"
    f"log-I · per-image mean±3σ  |  PSNR vs MERLIN  |  seed={SEED_FOR_VIS}  |  {arch_str}",
    fontsize=10,
    y=1.01,
)
plt.tight_layout(h_pad=1.2, w_pad=0.3)

if SAVE_FIGURES:
    out = PLOTS_DIR / f"Visualizations_Hamburg_{'_'.join(ARCHS_FOR_VIS)}.pdf"
    fig.savefig(out, bbox_inches="tight")
    print(f"Saved: {out}")
    export_manuscript(fig, "fig_qualitative_grid")
plt.show()

## Hamburg Tile — Close-up

Zoomed region `[{CROP_R0}:{CROP_R1}, {CROP_C0}:{CROP_C1}]` (set by `CROP_*` vars above).

In [ ]:
n_refs_cu = len(ref_panels)
n_lam_cu = len(sel_lambdas)
n_cols_cu = max(n_refs_cu, n_lam_cu)
n_rows_cu = 1 + 2 * len(ARCHS_FOR_VIS)

_ROW_HEADERS_CU = [(0, "References", "black")]
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    _suffix = f"\n{ARCH_LBL.get(_arch, _arch)}" if len(ARCHS_FOR_VIS) > 1 else ""
    _ROW_HEADERS_CU += [
        (1 + 2 * _i, f"FPGA{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["fpga"]),
        (2 + 2 * _i, f"GPU{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["gpu"]),
    ]

fig_cu, axes_cu = plt.subplots(
    n_rows_cu,
    n_cols_cu,
    figsize=(SUBPLOT_IN * n_cols_cu + 0.5, SUBPLOT_IN * n_rows_cu + 0.7),
    squeeze=False,
)

for _ri, _label, _color in _ROW_HEADERS_CU:
    axes_cu[_ri, 0].text(
        -0.12,
        0.5,
        _label,
        transform=axes_cu[_ri, 0].transAxes,
        rotation=90,
        va="center",
        ha="right",
        fontsize=9,
        fontweight="bold",
        color=_color,
    )

# Row 0: References (cropped)
for ci, (ref_name, ref_linA) in enumerate(ref_panels):
    _show(axes_cu[0, ci], _crop(_linA_to_logI(ref_linA)), ref_name)
for ci in range(n_refs_cu, n_cols_cu):
    axes_cu[0, ci].axis("off")

# FPGA + GPU rows (cropped)
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    for _j, backend in enumerate(["fpga", "gpu"]):
        row_idx = 1 + 2 * _i + _j
        color = BACKEND_COLORS[backend]
        for ci, lmbda in enumerate(sel_lambdas):
            data = tile_data[_arch].get(lmbda, {}).get(backend)
            if data is None:
                axes_cu[row_idx, ci].axis("off")
                axes_cu[row_idx, ci].text(
                    0.5,
                    0.5,
                    "N/A",
                    ha="center",
                    va="center",
                    transform=axes_cu[row_idx, ci].transAxes,
                    fontsize=10,
                    color="gray",
                )
                continue
            bpp_str = f"{data['bpp']:.3f}" if not np.isnan(data["bpp"]) else "N/A"
            subtitle = f"BPP={bpp_str}  PSNR={data['psnr_merlin']:.2f}dB"
            _show(
                axes_cu[row_idx, ci],
                _crop(data["logI"]),
                f"λ={int(lmbda)}",
                subtitle,
                border_color=color,
            )
        for ci in range(n_lam_cu, n_cols_cu):
            axes_cu[row_idx, ci].axis("off")

arch_str = " + ".join(ARCH_LBL.get(a, a) for a in ARCHS_FOR_VIS)
fig_cu.suptitle(
    f"Hamburg Tile — Close-up  rows [{CROP_R0}:{CROP_R1}]  cols [{CROP_C0}:{CROP_C1}]\n"
    f"log-I · per-image mean±3σ  |  PSNR vs MERLIN  |  seed={SEED_FOR_VIS}  |  {arch_str}",
    fontsize=10,
    y=1.01,
)
plt.tight_layout(h_pad=1.2, w_pad=0.3)

if SAVE_FIGURES:
    out_cu = PLOTS_DIR / f"Visualizations_Hamburg_closeup_{'_'.join(ARCHS_FOR_VIS)}.pdf"
    fig_cu.savefig(out_cu, bbox_inches="tight")
    print(f"Saved: {out_cu}")
plt.show()